# Tripitaka Website Diagnostic Tool

This notebook will help us understand the IFBC website structure and find ALL Tripitaka sections.

**Strategy:**
1. First, analyze the main category page structure
2. Look for alternative ways to find all posts (sitemap, archives, etc.)
3. Check if sections are organized differently than expected
4. Build a comprehensive list of all available sections

In [1]:
!pip install -q requests beautifulsoup4 lxml

In [2]:
import requests
from bs4 import BeautifulSoup
import re
from urllib.parse import urljoin, urlparse
import time

In [3]:
BASE_URL = 'https://download.ifbcnet.org'
CATEGORY_URL = 'https://download.ifbcnet.org/category/thripitaka-books/'

HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36'
}

## Step 1: Analyze Category Page Structure

In [4]:
# Fetch and analyze the main category page
resp = requests.get(CATEGORY_URL, headers=HEADERS)
soup = BeautifulSoup(resp.content, 'html.parser')

print('='*80)
print('CATEGORY PAGE ANALYSIS')
print('='*80)

# Check for pagination info
pagination = soup.find_all(class_=re.compile(r'paginat|nav'))
print(f'\nPagination elements found: {len(pagination)}')
for p in pagination[:3]:
    print(f'  {p.get("class")}: {p.get_text(strip=True)[:100]}')

# Look for post count or category info
category_info = soup.find(class_=re.compile(r'category|archive|count'))
if category_info:
    print(f'\nCategory info: {category_info.get_text(strip=True)[:200]}')

# Find all articles/posts
articles = soup.find_all('article')
print(f'\nArticles on page: {len(articles)}')

# Find all links that might be posts
all_links = soup.find_all('a', href=True)
print(f'Total links: {len(all_links)}')

tripitaka_links = [a['href'] for a in all_links if 'tripitaka' in a['href'].lower() or 'thripitaka' in a['href'].lower()]
print(f'\nLinks containing "tripitaka": {len(tripitaka_links)}')
for link in set(tripitaka_links):
    print(f'  - {link}')

CATEGORY PAGE ANALYSIS

Pagination elements found: 0

Category info: Skip to main contentHomeDhammaOur ProjectsDonationsAbout UsContact UssearchMenuHomesearchPress enter to begin your searchClose SearchCategoryත්‍රිපිටක පොත් වහන්සේ … ( Thripitaka books)Feb10ත්‍රිපිටකය 

Articles on page: 10
Total links: 99

Links containing "tripitaka": 16
  - https://download.ifbcnet.org/2015/02/10/%e0%b6%ad%e0%b7%8a%e2%80%8d%e0%b6%bb%e0%b7%92%e0%b6%b4%e0%b7%92%e0%b6%a7%e0%b6%9a%e0%b6%ba-%e0%b6%af%e0%b7%93%e0%b6%9d%e0%b6%b1%e0%b7%92%e0%b6%9a%e0%b7%8f%e0%b6%ba-thripitaka-digha-nikaya/
  - https://download.ifbcnet.org/category/thripitaka-books/page/2/
  - https://download.ifbcnet.org/category/thripitaka-books/


## Step 2: Check for Sitemap or Archive

In [5]:
# Try common sitemap URLs
sitemap_urls = [
    f'{BASE_URL}/sitemap.xml',
    f'{BASE_URL}/sitemap_index.xml',
    f'{BASE_URL}/post-sitemap.xml',
    f'{BASE_URL}/wp-sitemap.xml',
    f'{BASE_URL}/sitemap/',
]

print('='*80)
print('CHECKING FOR SITEMAPS')
print('='*80)

for url in sitemap_urls:
    try:
        resp = requests.get(url, headers=HEADERS, timeout=10)
        if resp.status_code == 200:
            print(f'\n✓ Found: {url}')
            soup = BeautifulSoup(resp.content, 'lxml-xml')
            locs = soup.find_all('loc')
            print(f'  URLs in sitemap: {len(locs)}')

            # Filter for Tripitaka posts
            tripitaka_urls = [loc.text for loc in locs if 'tripitaka' in loc.text.lower() or 'thripitaka' in loc.text.lower()]
            print(f'  Tripitaka URLs: {len(tripitaka_urls)}')

            if tripitaka_urls:
                print('\n  Sample URLs:')
                for url_text in tripitaka_urls[:10]:
                    print(f'    - {url_text}')

                if len(tripitaka_urls) > 10:
                    print(f'    ... and {len(tripitaka_urls) - 10} more')
    except Exception as e:
        print(f'✗ {url}: {e}')

CHECKING FOR SITEMAPS

✓ Found: https://download.ifbcnet.org/sitemap.xml
  URLs in sitemap: 2
  Tripitaka URLs: 0

✓ Found: https://download.ifbcnet.org/sitemap_index.xml
  URLs in sitemap: 3
  Tripitaka URLs: 0

✓ Found: https://download.ifbcnet.org/post-sitemap.xml
  URLs in sitemap: 156
  Tripitaka URLs: 1

  Sample URLs:
    - https://download.ifbcnet.org/2015/02/10/%e0%b6%ad%e0%b7%8a%e2%80%8d%e0%b6%bb%e0%b7%92%e0%b6%b4%e0%b7%92%e0%b6%a7%e0%b6%9a%e0%b6%ba-%e0%b6%af%e0%b7%93%e0%b6%9d%e0%b6%b1%e0%b7%92%e0%b6%9a%e0%b7%8f%e0%b6%ba-thripitaka-digha-nikaya/

✓ Found: https://download.ifbcnet.org/wp-sitemap.xml
  URLs in sitemap: 3
  Tripitaka URLs: 0


## Step 3: Look for Category Archives or Indexes

In [6]:
# Check the homepage for categories or indexes
print('='*80)
print('CHECKING HOMEPAGE FOR CATEGORIES')
print('='*80)

resp = requests.get(BASE_URL, headers=HEADERS)
soup = BeautifulSoup(resp.content, 'html.parser')

# Look for category widgets or menus
categories = soup.find_all(class_=re.compile(r'categor|menu|widget'))
print(f'\nFound {len(categories)} potential category containers')

# Find all category links
category_links = soup.find_all('a', href=re.compile(r'/category/'))
print(f'\nCategory links found: {len(category_links)}')
for link in category_links[:15]:
    print(f'  - {link.get_text(strip=True)}: {link["href"]}')

CHECKING HOMEPAGE FOR CATEGORIES

Found 34 potential category containers

Category links found: 16
  - : https://download.ifbcnet.org/category/thripitaka-books/
  - : https://download.ifbcnet.org/category/books-related-to-the-tipitaka/
  - : https://download.ifbcnet.org/category/old-books/
  - : https://download.ifbcnet.org/category/ven-rerukane-chandawimala-thero-books/
  - : https://download.ifbcnet.org/category/ven-balangoda-ananda-maithriya-thero-books/
  - : https://download.ifbcnet.org/category/ven-madihe-pannaseeha-thero-books/
  - : https://download.ifbcnet.org/category/ven-nauyane-ariyadhamma-maha-thero/
  - : https://download.ifbcnet.org/category/ven-matara-sri-gnanarama-thero-books/
  - : https://download.ifbcnet.org/category/ven-rrajagiriye-ariyagnaana-thero-books/
  - : https://download.ifbcnet.org/category/ven-katukurunde-gnanananda-thero-books/
  - : https://download.ifbcnet.org/category/ven-kiribathgoda-gnanananda-thero-books/
  - : https://download.ifbcnet.org/category

## Step 4: Check Navigation/Menu for Tripitaka Sections

In [7]:
print('='*80)
print('CHECKING NAVIGATION MENUS')
print('='*80)

# Look for nav elements
navs = soup.find_all('nav')
print(f'\nNavigation elements: {len(navs)}')

for i, nav in enumerate(navs, 1):
    print(f'\nNav {i}:')
    links = nav.find_all('a', href=True)
    for link in links[:10]:
        text = link.get_text(strip=True)
        href = link['href']
        if text and len(text) > 2:
            print(f'  - {text}: {href}')

CHECKING NAVIGATION MENUS

Navigation elements: 2

Nav 1:
  - Home: http://ifbcnet.org/
  - Dhamma: http://dhamma.ifbcnet.org/
  - Our Projects: http://ifbcnet.org/our-projects/
  - Donations: http://ifbcnet.org/donations/
  - About Us: http://ifbcnet.org/about-us/
  - Contact Us: http://ifbcnet.org/contact-us/

Nav 2:
  - Home: https://download.ifbcnet.org/
  - search: #searchbox


## Step 5: Check First Tripitaka Post for Structure

In [8]:
# Analyze the structure of the one post we found
POST_URL = 'https://download.ifbcnet.org/2015/02/10/%e0%b6%ad%e0%b7%8a%e2%80%8d%e0%b6%bb%e0%b7%92%e0%b6%b4%e0%b7%92%e0%b6%a7%e0%b6%9a%e0%b6%ba-%e0%b6%af%e0%b7%93%e0%b6%9d%e0%b6%b1%e0%b7%92%e0%b6%9a%e0%b7%8f%e0%b6%ba-thripitaka-digha-nikaya/'

print('='*80)
print('ANALYZING SAMPLE POST STRUCTURE')
print('='*80)

resp = requests.get(POST_URL, headers=HEADERS)
soup = BeautifulSoup(resp.content, 'html.parser')

# Look for related posts or navigation
related = soup.find_all(class_=re.compile(r'related|navigation|prev|next'))
print(f'\nRelated/Navigation elements: {len(related)}')

for elem in related[:5]:
    print(f'  Class: {elem.get("class")}')
    links = elem.find_all('a', href=True)
    for link in links[:3]:
        print(f'    - {link.get_text(strip=True)}: {link["href"]}')

# Look for category/tag links
print('\nCategories/Tags:')
cats = soup.find_all(['a'], rel=['category', 'tag'])
for cat in cats[:10]:
    print(f'  - {cat.get_text(strip=True)}: {cat.get("href")}')

ANALYZING SAMPLE POST STRUCTURE

Related/Navigation elements: 0

Categories/Tags:


## Step 6: Try Different Search Strategies

In [9]:
print('='*80)
print('TRYING ALTERNATIVE DISCOVERY METHODS')
print('='*80)

# Try searching or browsing archives
test_urls = [
    f'{BASE_URL}/2015/02/',  # Date archive
    f'{BASE_URL}/?s=tripitaka',  # Search
    f'{BASE_URL}/tag/tripitaka/',  # Tag
]

for url in test_urls:
    print(f'\nTrying: {url}')
    try:
        resp = requests.get(url, headers=HEADERS, timeout=10)
        if resp.status_code == 200:
            soup = BeautifulSoup(resp.content, 'html.parser')
            articles = soup.find_all('article')
            print(f'  ✓ Found {len(articles)} articles')

            # Look for Tripitaka posts
            for article in articles[:5]:
                title_elem = article.find(['h1', 'h2', 'h3'])
                if title_elem:
                    link = title_elem.find('a', href=True)
                    if link:
                        print(f'    - {title_elem.get_text(strip=True)}')
                        print(f'      {link["href"]}')
        else:
            print(f'  ✗ Status: {resp.status_code}')
    except Exception as e:
        print(f'  ✗ Error: {e}')

    time.sleep(1)

TRYING ALTERNATIVE DISCOVERY METHODS

Trying: https://download.ifbcnet.org/2015/02/
  ✓ Found 10 articles
    - බුද්ධ වන්දනා ක්‍රමය – පූජ්‍ය නාඋයනේ අරියධම්ම හිමි (Buddha wandana Kramaya – Ven Nauyane Ariyadhamma Maha Thero )
      https://download.ifbcnet.org/2015/02/27/%e0%b6%b6%e0%b7%94%e0%b6%af%e0%b7%8a%e0%b6%b0-%e0%b7%80%e0%b6%b1%e0%b7%8a%e0%b6%af%e0%b6%b1%e0%b7%8f-%e0%b6%9a%e0%b7%8a%e2%80%8d%e0%b6%bb%e0%b6%b8%e0%b6%ba-%e0%b6%b4%e0%b7%96%e0%b6%a2/
    - බෝධි වන්දනා – පූජ්‍ය නාඋයනේ අරියධම්ම හිමි (Bodhi wandana – Ven Nauyane Ariyadhamma Maha Thero )
      https://download.ifbcnet.org/2015/02/27/%e0%b6%b6%e0%b7%9d%e0%b6%b0%e0%b7%92-%e0%b7%80%e0%b6%b1%e0%b7%8a%e0%b6%af%e0%b6%b1%e0%b7%8f-%e0%b6%b4%e0%b7%96%e0%b6%a2%e0%b7%8a%e2%80%8d%e0%b6%ba-%e0%b6%b1%e0%b7%8f%e0%b6%8b%e0%b6%ba/
    - භාවනා – පූජ්‍ය නාඋයනේ අරියධම්ම හිමි (Bhawana – Ven Nauyane Ariyadhamma Maha Thero )
      https://download.ifbcnet.org/2015/02/26/%e0%b6%b7%e0%b7%8f%e0%b7%80%e0%b6%b1%e0%b7%8f-%e0%b6%b4%e0%b7%96%e0%b6%a2%e

## Step 7: Manual List Based on Tripitaka Structure

Based on the image you shared, it looks like the sidebar shows other sections. Let me check the sidebar of the category page.

In [10]:
print('='*80)
print('ANALYZING SIDEBAR FOR SECTION LINKS')
print('='*80)

# Re-fetch category page and look at sidebar
resp = requests.get(CATEGORY_URL, headers=HEADERS)
soup = BeautifulSoup(resp.content, 'html.parser')

# Look for sidebar or widget areas
sidebars = soup.find_all(['aside', 'div'], class_=re.compile(r'sidebar|widget|side'))
print(f'\nSidebar elements found: {len(sidebars)}')

all_section_links = []

for i, sidebar in enumerate(sidebars, 1):
    print(f'\n--- Sidebar {i} ---')

    # Find all links in sidebar
    links = sidebar.find_all('a', href=True)
    print(f'Links in sidebar: {len(links)}')

    for link in links:
        href = link['href']
        text = link.get_text(strip=True)

        # Check if it's a Tripitaka section
        if any(kw in href.lower() for kw in ['tripitaka', 'thripitaka', 'tipitaka']):
            all_section_links.append((text, href))
            print(f'  ✓ {text}')
            print(f'    {href}')

print(f'\n{'='*80}')
print(f'TOTAL TRIPITAKA LINKS FOUND IN SIDEBARS: {len(all_section_links)}')
print('='*80)

ANALYZING SIDEBAR FOR SECTION LINKS

Sidebar elements found: 4

--- Sidebar 1 ---
Links in sidebar: 1

--- Sidebar 2 ---
Links in sidebar: 16
  ✓ ත්‍රිපිටක ආශ්‍රිත පොත්Books Related to the Tipitaka
    https://download.ifbcnet.org/category/books-related-to-the-tipitaka/
  ✓ ත්‍රිපිටක පොත් වහන්සේ …Thripitaka books
    https://download.ifbcnet.org/category/thripitaka-books/

--- Sidebar 3 ---
Links in sidebar: 6

--- Sidebar 4 ---
Links in sidebar: 0

TOTAL TRIPITAKA LINKS FOUND IN SIDEBARS: 2


## Summary and Recommendations

Run all cells above to understand:
1. How many pages exist in the category
2. Whether there's a sitemap with all posts
3. If sections are listed in sidebars or navigation
4. Alternative ways to access all Tripitaka content

Based on the results, we'll create a targeted scraper.